# Train Smart Waste YOLO (Google Colab)

Use a GPU runtime. This notebook fine-tunes a lightweight pretrained YOLO checkpoint on GARBAGE CLASSIFICATION 3. It does **not** train from random initialization unless you set `PRETRAINED = False`.

The notebook rebuilds a stratified 70/20/10 split after checking near-duplicates. The raw source split is not used because glass is absent from its test set. Review `docs/dataset_examples.jpg` before training. After training, download `models/best.pt` into the local Streamlit project and run evaluation on the untouched stratified test split.

In [ ]:
import os, platform, shutil, subprocess, sys, time
from pathlib import Path

print(platform.platform())
print(sys.version)
try:
    import torch
    print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))
except Exception as exc:
    print('torch not imported yet', exc)

ROOT = Path('/content/smart-waste-assistant')
if not (ROOT / 'scripts').exists():
    ROOT = Path('/content/smart-waste-assistant-clone')
    if not (ROOT / 'scripts').exists():
        subprocess.run(['git', 'clone', 'https://github.com/Sandip-Bhalekar/smart-waste-assistant.git', str(ROOT)], check=True)
os.chdir(ROOT)
print('cwd', os.getcwd())

If this notebook is running inside the cloned project, skip the copy cell. Otherwise upload the project or clone it next to this notebook.

In [ ]:
%pip -q install ultralytics requests pyyaml pandas matplotlib seaborn pillow opencv-python-headless

In [ ]:
import torch, yaml
from ultralytics import YOLO

PRETRAINED = True
EPOCHS = 60
IMGSZ = 416
BATCH = 16 if torch.cuda.is_available() else 4
DEVICE = '0' if torch.cuda.is_available() else 'cpu'
MODEL = 'yolo11n.pt'
print({'pretrained': PRETRAINED, 'epochs': EPOCHS, 'imgsz': IMGSZ, 'batch': BATCH, 'device': DEVICE, 'model': MODEL})

In [ ]:
# Download the source data, inspect it, then build the leakage-checked multilabel split.
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd()))
if Path('scripts/download_dataset.py').exists():
    %run scripts/download_dataset.py
    %run scripts/inspect_dataset.py
    %run scripts/prepare_stratified_dataset.py --output data/processed/garbage-classification-3-stratified-v2
    %run scripts/validate_dataset.py --processed-root data/processed/garbage-classification-3-stratified-v2
else:
    raise SystemExit('Upload or clone the smart-waste-assistant project so scripts/ is available.')

In [ ]:
started = time.time()
if PRETRAINED:
    model = YOLO(MODEL)
else:
    print('Random initialization requested. Expect weaker results on this dataset size.')
    model = YOLO(MODEL.replace('.pt', '.yaml'))

results = model.train(
    data='data/processed/garbage-classification-3-stratified-v2/waste.yaml',
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project='runs/detect',
    name='waste-yolo-colab',
    exist_ok=True,
    seed=42,
    patience=15,
    pretrained=PRETRAINED,
    optimizer='AdamW',
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    degrees=8.0,
    translate=0.08,
    scale=0.4,
    fliplr=0.5,
    flipud=0.0,
    plots=True,
)
print('train_seconds', round(time.time() - started, 2))
print('save_dir', results.save_dir)

In [ ]:
from pathlib import Path
best = Path(results.save_dir) / 'weights' / 'best.pt'
Path('models').mkdir(exist_ok=True)
shutil.copy2(best, 'models/best.pt')
print('copied', best, '-> models/best.pt', 'size_mb', round(best.stat().st_size / 1024 / 1024, 3))
print('Download models/best.pt to your laptop before running Streamlit.')

In [ ]:
# Held-out test evaluation. Do not use these numbers to pick checkpoints.
%run scripts/evaluate.py --weights models/best.pt --data data/processed/garbage-classification-3-stratified-v2/waste.yaml --split test --imgsz 416